In [1]:
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

import psutil
import math
import gc
from collections import Counter
import re


import os, random
import posixpath
from pyhdas.frequency import spectrogram, add_db, energy, power_spectrum
from pyhdas.aggregate import quantile
from pyhdas.aragon import concat_raw_data, aragon_select_files

from datetime import timedelta

In [ ]:
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
def calculate_low_frequency_spectrum(
    strain,
    stride=16,
    low_frequency_cutoff=50,
    low_frequency_min=0.1,
):
    """Calculate low-frequency spectrum from strain signal

    Parameters
    ----------
    strain : xarray
        DAS strain measurement time series
    stride: integer, number of positions to calculate at once
        Using a higher stride will be faster on computers with a lot of RAM.
    low_frequency_cutoff : float, default 50
        Cutoff of the frequencies to discard when saving the low frequency
        outputs.
    low_frequency_min : float, default 0.1
        Minimal frequency for the power_spectrum of the low
        frequencies output

    Returns
    -------
    xarray
        low-frequency spectrum
    """

    all_spectra = []
    for j in range(0, len(strain.position), stride):
        low_freq_spectrum_ = power_spectrum(
            strain.isel(position=slice(j, j + stride)),
            min_frequency=low_frequency_min,
            keep_time_dim=True,
        )
        low_freq_spectrum_ = low_freq_spectrum_.sel(
            freq=slice(
                low_frequency_cutoff,
            )
        )
        all_spectra.append(low_freq_spectrum_)

    low_freq_spectrum = xr.concat(all_spectra, dim="position")

    return low_freq_spectrum

In [ ]:
def normal_low_freq_plots(files, dir):

    all_rows = []

    for file in files:

        filepath = [posixpath.join(dir, file)]

        # open file one by one
        ds_raw = concat_raw_data(filepath)

        # select locations 4210-4260
        poi = np.arange(4200, 4300, 10)

        ds_raw = ds_raw.sel(position=poi)
        ds_lowfreq_spect = calculate_low_frequency_spectrum(ds_raw)
        
        fig,ax = plt.subplots(figsize=(15,6))
        ds_lowfreq_spect.Pxx_dB.isel(time=0).plot(x="position", y="freq", ax=ax, vmin=-40, vmax=40, cmap="magma")
        ax.set_title(f"lowfrequency position spectrogram")
        
        # save in the normal data directory
        output_folder = Path(f"plots/new_normal_low_freq_spectrograms")
        output_folder.mkdir(parents=True, exist_ok=True)

        plot_filename = f"{file[:-4]}_high_FP.png"
        plt.savefig(output_folder / plot_filename, bbox_inches='tight')

        plt.close(fig)

        row = dict()
        row["start"] = file[:-4]
        row["end"] = None
        row["event_label"] = "normal"

        all_rows.append(row)
        
        print(f"Done with {file}")
        del ds_raw, ds_lowfreq_spect
        gc.collect()

    if all_rows:
        final_df = pd.DataFrame(all_rows)
        output_folder = Path("low_freq_data_csv")
        output_folder.mkdir(parents=True, exist_ok=True)
        final_df.to_csv(output_folder / "annotations_normal_hours.csv", index=False)

In [ ]:
# select 200 random files from the directory
normal_data_dir = "data/19"
random.seed(58)
files = random.sample(os.listdir(normal_data_dir), 100)
normal_low_freq_plots(files, normal_data_dir)

In [ ]:
def create_event_annotations():

    all_rows = []
    location_columns = np.arange(4220, 4270, 10)

    events = pd.read_csv("events_table.csv", parse_dates=["start", "end"])

    for row, event in events.iterrows(): 

        start, end, poi, label = event["start"], event["end"], event["poi"], event["label_anon"]

        start = pd.Timestamp(start)
        end = pd.Timestamp(end)

        # Skip events that are too short or too long
        event_duration = (end - start).total_seconds()
        if event_duration > 300 or event_duration <= 1:
            continue

        end = end.tz_localize("UTC")
        start = start.tz_localize("UTC")

        end_file = start
        dir_data = Path(fr"data/{end.day}")


        while end_file < end:

            end_file = start + pd.Timedelta(seconds=60)

            print(start, end_file)
            row = dict()
            row["start"] = start
            row["end"] = end_file
            row["event_label"] = label
            
            for loc in location_columns:
                row[str(loc)] = None

            all_rows.append(row)
            
            start = end_file


    if all_rows:
            final_df = pd.DataFrame(all_rows)
            output_folder = Path("low_freq_data_csv")
            output_folder.mkdir(parents=True, exist_ok=True)
            final_df.to_csv(output_folder / "annotations_test.csv", index=False)

In [3]:
def get_low_freq_data(csv_directory, output_file_path, mode='annotated', position_table_path='position_table.csv', excluded_positions=None):
    """
    Parameters:
        csv_directory: folder with annotation CSVs
        output_file_path: full path to the output CSV file
        mode: 'annotated' or 'quiet'
        position_table_path: path to position_table.csv (used in quiet mode)
        excluded_positions: list of position_fiber values to exclude in quiet mode
    """

    output_folder = Path(output_file_path).parent
    output_folder.mkdir(parents=True, exist_ok=True)
    output_file = Path(output_file_path)

    if mode == 'quiet':
        if excluded_positions is None:
            excluded_positions = []

        position_table = pd.read_csv(position_table_path)
        selected = position_table[
            (position_table['location_name'] == 'quiet') &
            (~position_table['position_fiber'].isin(excluded_positions))
        ]
        locs = selected['position_fiber'].values
    else:
        locs = None  # will be determined per file from annotation

    for annotation_file in os.listdir(csv_directory):
        csv_path = os.path.join(csv_directory, annotation_file)
        annotations_df = pd.read_csv(csv_path)

        print(f"Working with file: {annotation_file}")

        for idx, row in annotations_df.iterrows():
            if pd.notna(row['end']):
                start = pd.Timestamp(row['start'])
                end = pd.Timestamp(row['end'])
                day = start.day
                dir_data = Path(f"data/{day}")
                file_list = list(aragon_select_files(dir_data, start, end, extension="bin"))
                start = start.tz_localize(None)
                end = end.tz_localize(None)
            else:
                dir_data = Path("data/19")
                file_list = [posixpath.join(dir_data, f"{row['start']}.bin")]

            try:
                ds_raw = concat_raw_data(file_list)
            except Exception as e:
                print(f"Failed to load raw data for row {idx}: {e}")
                continue

            if mode == 'annotated':
                location_columns = annotations_df.columns[3:]
            else:
                location_columns = locs

            for col in location_columns:
                if mode == 'annotated':
                    cell_value = row[col]
                    if cell_value not in [0, 1]:
                        continue
                else:
                    cell_value = 0  # default for quiet mode

                location = [int(col)]

                try:
                    if len(file_list) > 1:
                        ds_raw_slice = ds_raw.sel(time=slice(start, end), position=location)
                    else:
                        ds_raw_slice = ds_raw.sel(position=location)
                except Exception as e:
                    print(f"Failed slicing for location {location} at row {idx}: {e}")
                    continue

                ds_lowfreq_spect = calculate_low_frequency_spectrum(ds_raw_slice)
                pxx = ds_lowfreq_spect.Pxx_dB.values

                flat_values = pxx.flatten()
                col_names = [f"f{fi}" for fi in range(len(flat_values))]

                r = dict(zip(col_names, flat_values))
                r["start"] = row['start']
                r["end"] = row['end']
                r["location"] = location
                r["event_label"] = row['event_label']
                r["label"] = cell_value

                row_df = pd.DataFrame([r])
                write_header = not output_file.exists()
                row_df.to_csv(output_file, mode='a', index=False, header=write_header)

                del ds_raw_slice, ds_lowfreq_spect

            del ds_raw
            gc.collect()
            print(f"Done with row {idx}")


In [ ]:
get_low_freq_data(
    csv_directory="annotations",
    output_file_path="low_freq_data/low_freq_test_normal_day.csv",
    mode='quiet',
    excluded_positions=[1550, 1600]
)